# Mini-projet : Assistant d’analyse des sentiments avec optimisation de BERT

Bienvenue au mini-projet du dernier jour !

Dans cet atelier, vous perfectionnerez vos compétences `bert-base-uncased` en matière de critiques de films, évaluerez le modèle et l'appliquerez à des scénarios concrets de service client. Chaque section explique la raison d'être de chaque étape, les points à observer et propose des illustrations pour faciliter la compréhension du concept par vos apprenants.

## Prérequis et mise en place de l'histoire
**Scénario** : L’équipe d’analyse du support souhaite disposer d’un signal fiable de ressenti pour les commentaires longs afin de pouvoir remonter les problèmes des clients mécontents avant qu’ils ne se désabonnent.
**Configuration requise** : Python 3.9+, un environnement d’exécution compatible GPU (Colab, Kaggle ou une machine équipée d’un GPU local) et environ 6 Go de VRAM libre. L’utilisation du CPU uniquement est possible, mais l’entraînement sera plus long.
**Packages** : `tensorflow`, `tensorflow-datasets`, `transformers`, `accelerate`, et `evaluate`, qui sont tous apparus plus tôt dans le cours, vous avez donc déjà utilisé ces outils auparavant.

---

### Installation des bibliothèques nécessaires

In [ ]:
# Run once in a fresh environment
pip install -q tensorflow tensorflow-datasets transformers accelerate evaluate

print("\n🔍 Vous remarquerez ici que nous réutilisons exactement la même chaîne d'outils que les jours 3 et 4. Cela renforce la continuité et nous permet de nous concentrer sur le nouveau flux de travail plutôt que sur de nouvelles bibliothèques.")

### Vérification des importations et du matériel
Nous commençons toujours par vérifier les versions et le matériel. Si un apprenant voit cela `GPU devices: []`, il sait immédiatement qu'il doit changer d'environnement d'exécution (lorsque nous utilisons Google Colab, aucune autre installation ni configuration n'est nécessaire).

In [ ]:
import platform
import tensorflow as tf
import tensorflow_datasets as tfds
from transformers import BertTokenizer, TFBertForSequenceClassification

print("Python version      :", platform.python_version())
print("TensorFlow version  :", tf.__version__)
print("GPU devices detected:", tf.config.list_physical_devices('GPU'))

### Charger l'ensemble de données des critiques IMDB
Nous utilisons IMDb car ses avis sont équilibrés (25 000 positifs / 25 000 négatifs) et déjà segmentés. Les apprenants devraient le reconnaître grâce aux exemples d'analyse de sentiments précédents.

💬 Ici, vous remarquerez que TFDS renvoie à la fois les objets du jeu de données et leurs métadonnées. Notez que cela `as_supervised=True` produit `(text, label)` des paires, exactement ce que notre modèle attend.

In [ ]:
(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=(tfds.Split.TRAIN, tfds.Split.TEST),
    as_supervised=True,
    with_info=True
)
print(ds_info)

Jetez un coup d'œil rapide à ces exemples pour concrétiser vos idées :

In [ ]:
for text, label in ds_train.take(2):
    print("Label:", "Positive" if label.numpy() else "Negative")
    print(text.numpy().decode()[:250], "...\n")

### Configuration du tokenizer et du pipeline de données
BERT utilise la tokenisation WordPiece pour gérer les mots rares ou inconnus en les décomposant en sous-mots, garantissant ainsi une couverture complète et une taille de vocabulaire optimale. Il ajoute `[CLS]` des `[SEP]` tokens pour marquer les limites des phrases et permettre des tâches telles que la classification et la modélisation de paires de phrases. Les masques d'attention indiquent au modèle quels tokens sont valides et lesquels sont des tokens de remplissage, assurant ainsi que l'attention ne soit calculée que sur les entrées valides.

🧠 Nous importons ce tokenizer pour réutiliser le même vocabulaire que celui appris par le modèle de base en 2018.

Ensuite, convertissez les octets bruts en identifiants de jetons, masques d'attention et identifiants de segments.

🗒️ N'oubliez pas que cela `tf.py_function` nous permet de conserver la logique de tokenisation Hugging Face au sein d'un pipeline TensorFlow, évitant ainsi de manipuler manuellement les tableaux NumPy. De plus, le brassage et le préchargement stabilisent le débit d'entraînement.

In [ ]:
MAX_LENGTH = 256   # trim or pad every review to 256 tokens so batches align
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
print("Tokenizer loaded:", tokenizer.name_or_path)

def encode_review(review_input):
    if isinstance(review_input, bytes):
        review_text = review_input.decode("utf-8")
    elif hasattr(review_input, "numpy"):
        review_text = review_input.numpy().decode("utf-8")
    else:
        review_text = str(review_input)

    return tokenizer.encode_plus(
        review_text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
    )

def tf_encode(text, label):
    encoded = tf.py_function(
        func=lambda t: list(encode_review(t).values()),
        inp=[text],
        Tout=[tf.int32, tf.int32, tf.int32]
    )
    return {
        "input_ids": encoded[0],
        "attention_mask": encoded[1],
        "token_type_ids": encoded[2]
    }, label

def prepare_dataset(dataset):
    return (
        dataset
        .map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE)
        .shuffle(2000)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

train_ds = prepare_dataset(ds_train)
test_ds  = prepare_dataset(ds_test)

### Initialiser le modèle de réglage fin
Nous chargeons maintenant `TFBertForSequenceClassification`, qui regroupe déjà l'encodeur et la tête de classification.

💡 Nous importons ce modèle afin de réutiliser les 110 millions de paramètres appris sur BooksCorpus et Wikipédia. Nous effectuons un réglage fin uniquement sur quelques époques, ce qui explique les taux d'apprentissage de l'ordre de 2e-5.

In [ ]:
model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    use_safetensors=False
)

optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-8)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]

model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
model.summary()

### Former et superviser
Sur un GPU T4 (celui utilisé dans Google Colab), deux époques prennent environ 15 minutes. Veuillez surveiller la précision des calculs d'entraînement et de validation.

📈 Indiquez comment le plateau de précision de la validation signale le moment d'arrêter. Encouragez également la publication de captures d'écran des courbes d'apprentissage des portefeuilles.

In [ ]:
EPOCHS = 2  # increase to 3 if time allows
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=test_ds # Use test_ds for validation data during training
)

### Évaluer sur l'ensemble de test mis de côté
Même si `model.fit` des indicateurs de validation sont déjà fournis, nous relançons l'évaluation sur le pipeline de test intact afin de simuler l'assurance qualité en production.

✅ *Vous pourrez ici constater si la précision dépasse le seuil de référence de ~0,90 en classe.

#À faire : Utiliser ceci pour discuter des taux d’erreur acceptables pour les équipes de support réelles.*

In [ ]:
eval_metrics = model.evaluate(test_ds)
print(f"Test Loss: {eval_metrics[0]:.4f}, Test Accuracy: {eval_metrics[1]:.4f}")

### Créer un assistant d'inférence réutilisable
Intégrez le tout dans une fonction afin de pouvoir coller de véritables transcriptions de justificatifs et obtenir un score instantané.

Pour l'exemple précédent, vous devriez voir `Prediction: Positive (confidence=0.5)` slightly more or less.

🧭 Les scores de confiance sont essentiels pour décider s'il faut répondre automatiquement ou faire appel à un humain.

In [ ]:
import numpy as np

def predict_sentiment(text: str):
    # Encode the input text
    encoded_input = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        return_tensors="tf"  # Return TensorFlow tensors
    )

    # Make prediction
    outputs = model(encoded_input)
    logits = outputs.logits
    probabilities = tf.nn.softmax(logits, axis=-1).numpy()[0]

    # Determine the predicted label and confidence
    predicted_class_id = np.argmax(probabilities)
    label = "Positive" if predicted_class_id == 1 else "Negative"
    confidence = probabilities[predicted_class_id]

    return label, float(confidence)

custom_sentence = "The onboarding emails were confusing, but the agent fixed everything politely."
label, confidence = predict_sentiment(custom_sentence)
print(f"Prediction: {label} (confidence={confidence:.3f})")

custom_sentence_negative = "I am extremely unhappy with the service, it was absolutely terrible and a waste of my time."
label_neg, confidence_neg = predict_sentiment(custom_sentence_negative)
print(f"Prediction: {label_neg} (confidence={confidence_neg:.3f})")

custom_sentence_neutral = "The product arrived on time."
label_neu, confidence_neu = predict_sentiment(custom_sentence_neutral)
print(f"Prediction: {label_neu} (confidence={confidence_neu:.3f})")

## Réflexion et prochaines étapes

**Pourquoi le réglage fin est important** : Vous avez réutilisé un point de contrôle public pour atteindre une précision supérieure à 90 % avec un minimum de données.
**Compétences transférables** : Tout ce qui précède s’applique également aux tâches de classification dans les domaines des RH, du juridique ou de l’analyse de produits.
**Ce que vous pouvez faire avec ceci** : adaptation de domaine (collecte des e-mails de votre entreprise), points de contrôle multilingues (DistilBERT multilingue, XLM-R) et surveillance (enregistrement des dérives de données, création de tableaux de bord).

### Répondre aux questions de réflexion suivantes dans les cellules Markdown de votre cahier :

---

#### Quel levier (nettoyage des données, hyperparamètres, nombre d'époques) a le plus amélioré les résultats ?

Le levier qui a le plus amélioré les résultats est l'utilisation d'un **modèle pré-entraîné (BERT-base-uncased)** et son **réglage fin (fine-tuning)** sur l'ensemble de données spécifique (IMDb). Plutôt que de construire un modèle à partir de zéro, l'exploitation des connaissances linguistiques déjà acquises par BERT sur de vastes corpus de texte permet d'atteindre une précision élevée avec un nombre limité d'époques et de données spécifiques à la tâche. Les hyperparamètres comme le taux d'apprentissage faible (2e-5) et le nombre d'époques sont également cruciaux, mais ils optimisent l'apprentissage à partir de la base solide fournie par le modèle pré-entraîné, plutôt que de créer l'amélioration fondamentale.

#### Où ajouteriez-vous des garde-fous avant de déployer ce signal de sentiment en production ?

Avant de déployer ce signal de sentiment en production, j'ajouterais les garde-fous suivants :

1.  **Seuil de confiance** : Définir un seuil de confiance (par exemple, 0.7 ou 0.8) en dessous duquel le sentiment n'est pas automatiquement classifié. Les cas avec une faible confiance seraient marqués pour une révision humaine. Cela réduit le risque de classification erronée pour les textes ambigus ou complexes.
2.  **Détection de l'ambiguïté/Neutralité** : Ajouter une logique pour identifier les critiques véritablement neutres ou celles contenant des émotions mixtes (positives et négatives à la fois). Un modèle binaire (positif/négatif) peut forcer une classification qui ne reflète pas la réalité, ce qui est particulièrement problématique pour les critiques de support client.
3.  **Surveillance de la dérive des données (Data Drift)** : Mettre en place un système de surveillance pour détecter si la distribution des avis clients change avec le temps (par exemple, un nouveau vocabulaire, des expressions argotiques, des changements de produit). Si une dérive significative est détectée, le modèle pourrait nécessiter un nouveau réglage fin ou un réentraînement.
4.  **Tests A/B et déploiement progressif** : Déployer le modèle initialement auprès d'un petit sous-ensemble d'utilisateurs ou de cas, puis surveiller ses performances et l'impact sur les flux de travail avant un déploiement complet.
5.  **Explicabilité des résultats** : Fournir une certaine forme d'explication ou de mise en évidence des mots/phrases qui ont le plus contribué à la prédiction du sentiment. Cela aiderait les agents de support à comprendre pourquoi une classification a été faite et à intervenir plus efficacement.

#### Quels sont les acteurs qui en bénéficient le plus (responsable du support, chef de produit, responsable de la conformité) ?

Les acteurs qui en bénéficieraient le plus sont :

1.  **Responsable du Support Client** : C'est le bénéficiaire principal, car le signal de sentiment permet d'identifier rapidement les clients mécontents ou les problèmes urgents. Cela aide à prioriser les tickets, à réduire les temps de réponse pour les cas critiques et, in fine, à améliorer la satisfaction client et à réduire le taux de désabonnement.
2.  **Chef de Produit** : Ce signal peut fournir des informations précieuses sur les points faibles du produit ou les fonctionnalités appréciées. En analysant les tendances de sentiment liées à des aspects spécifiques du produit, les chefs de produit peuvent identifier les domaines à améliorer ou les succès à reproduire, orientant ainsi la feuille de route du produit.
3.  **Équipe Marketing/Communication** : Comprendre le sentiment général des clients peut éclairer les stratégies de messagerie, identifier les ambassadeurs de la marque et réagir rapidement aux crises de réputation en ligne.

Le **Responsable de la Conformité** pourrait également en bénéficier, bien que de manière moins directe. Si les critiques de sentiment révèlent des problèmes liés à la conformité réglementaire (par exemple, des plaintes concernant des pratiques trompeuses ou des violations de la vie privée), le signal de sentiment pourrait alerter sur ces problèmes pour une enquête plus approfondie.